# 桃園市空品資料爬蟲

依據專案計畫書，範圍為**資料蒐集與整理階段**（不含模型訓練）。

1. 歷史資料整理：寬表 CSV → 長表時間序列
2. 即時資料爬蟲：環境部 AQI API（`aqx_p_432`）+ 中央氣象署氣象 API（`O-A0001-001`，補氣溫/濕度/雨量）
3. 檢視整理後的資料集

執行前請確認專案根目錄的 `.env` 已設定 `MOENV_API_KEY` 與 `CWA_API_KEY`。

## 0. 環境設定

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

import pandas as pd
pd.set_option("display.max_columns", 50)

import config
import historical_preprocess
import realtime_crawler
import weather_crawler

print("專案根目錄：", PROJECT_ROOT)
print("桃園地區測站：", config.TAOYUAN_STATIONS)

/Users/mark.yu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


專案根目錄： /Users/mark.yu/Desktop/學生教材/nschool_2607_air_quality_monitor
桃園地區測站： ['桃園', '平鎮', '大園', '觀音', '龍潭']


## 1. 歷史資料整理

將 `data/raw_historical/` 內的原始寬表 CSV（桃園、平鎮、大園、觀音、龍潭）
melt/pivot 成長表時間序列，並補值、衍生時間特徵。

若尚未放入原始 CSV，此步驟會拋出 `FileNotFoundError`，可先跳過直接做第 2 節。

In [2]:
historical_csv_files = sorted(config.RAW_HISTORICAL_DIR.glob("*.csv"))
print(f"找到 {len(historical_csv_files)} 個歷史 CSV：", [p.name for p in historical_csv_files])

找到 0 個歷史 CSV： []


In [3]:
if historical_csv_files:
    historical_df = historical_preprocess.build_historical_dataset()
    out_path = config.PROCESSED_DIR / "historical_long.csv"
    config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    historical_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"完成，共 {len(historical_df)} 列，輸出至 {out_path}")
    display(historical_df.head())
else:
    historical_df = None
    print("尚未放入歷史 CSV，略過此步驟")

尚未放入歷史 CSV，略過此步驟


## 2. 即時資料爬蟲

呼叫環境部 AQI API 取得桃園 5 站當下觀測值，並用中央氣象署 API 補上氣溫/濕度/雨量
（`aqx_p_432` 本身不含氣象欄位）。

In [4]:
realtime_crawler.run_once()

2026-07-20 13:33:18,743 [INFO] 開始拉取即時空氣品質資料...
2026-07-20 13:33:19,027 [INFO] 原始資料存檔：/Users/mark.yu/Desktop/學生教材/nschool_2607_air_quality_monitor/data/realtime_raw/aqx_p_432_20260720_133319.json
2026-07-20 13:33:19,262 [INFO] 已累加寫入 /Users/mark.yu/Desktop/學生教材/nschool_2607_air_quality_monitor/data/processed/realtime_long.csv（累計 5 列，本次新增/更新 5 列）
2026-07-20 13:33:19,262 [INFO] 本次拉取完成，桃園地區有效筆數：5


In [5]:
realtime_path = config.PROCESSED_DIR / "realtime_long.csv"
realtime_df = pd.read_csv(realtime_path, parse_dates=["timestamp"])
display(realtime_df)

,測站,CO,NO2,NO,NOx,SO2,O3,PM10,PM2.5,WIND_SPEED,WIND_DIREC,經度,緯度,timestamp,AMB_TEMP,RH,RAINFALL,weather_obs_time,day_of_year,hour_of_day
0,大園,0.10,2,0.5,2.5,0.3,34,10,1,4.1,250,121.202515,25.061003,2026-07-20 13:00:00,33.8,59.0,0.0,2026-07-20T13:00:00+08:00,201,13
1,平鎮,0.13,5,1.0,6.3,0.6,39,12,2,3.0,273,121.203990,24.952785,2026-07-20 13:00:00,31.3,68.0,0.0,2026-07-20T13:00:00+08:00,201,13
2,桃園,0.14,5,1.6,7.6,0.1,46,10,7,2.2,266,121.305010,24.994710,2026-07-20 13:00:00,35.0,45.0,0.0,2026-07-20T13:00:00+08:00,201,13
3,觀音,0.06,1,0.8,2.5,0.7,34,9,5,4.1,252,121.082830,25.035568,2026-07-20 13:00:00,31.2,62.0,0.0,2026-07-20T13:00:00+08:00,201,13
4,龍潭,0.13,3,0.5,4.0,1.1,40,10,10,4.9,247,121.216460,24.864000,2026-07-20 13:00:00,32.0,54.0,0.5,2026-07-20T13:00:00+08:00,201,13


## 3. 測站 - 氣象站對照檢查

確認每個空品測站配對到的最近氣象站是否合理（距離是否過遠）。

In [6]:
import json
mapping_path = weather_crawler.STATION_MAPPING_CACHE
if mapping_path.exists():
    mapping = json.loads(mapping_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(mapping).T)
else:
    print("尚無快取對照表，需先執行第 2 節")

,cwa_station,distance_km
桃園,桃園,1.85
大園,觀音,6.23
觀音,新興坑尾,3.58
平鎮,國一高架N063K,0.82
龍潭,龍潭,0.84


## 4. （選用）常駐排程

若要每小時自動拉取一次即時資料，於終端機執行：

```bash
python src/realtime_crawler.py --loop
```

或改用系統 cron，於每小時 15 分觸發 `python src/realtime_crawler.py`。